


# Post-Training LLMs with DPO and GRPO

In this lab, we improve a language model with **two post-training techniques**:

### Learning Objectives
- What **Direct Preference Optimization (DPO)** is and how it helps models choose better answers.
- What **Group Relative Policy Optimization (GRPO)** is and how it improves performance on tasks with verifiable answers (math, code, logic).
- How to train a small model with both methods on a single GPU.
- How to **observe changes in model behavior** after fine-tuning.


### Tools and Data
- **Hugging Face Transformers** to load and run models
- **TRL (Transformer Reinforcement Learning)** library by Hugging Face for DPO and GRPO
- **A subset** of [French Orca DPO pairs](https://huggingface.co/datasets/AIffl/french_orca_dpo_pairs), a French translation of the `Intel/orca_dpo_pairs` preference dataset, for DPO
- **A subset** of [GSM8K](https://huggingface.co/datasets/openai/gsm8k), grade-school math word problems, for GRPO
- A **Colab T4 GPU**: the models are small enough to be trained on it

### Useful links:
- [Hugging Face DPO Trainer](https://huggingface.co/docs/trl/dpo_trainer)
- [Hugging Face GRPO Trainer](https://huggingface.co/docs/trl/grpo_trainer)


# Setup

This lab is designed for a free Google Colab **T4 GPU** (Runtime > Change runtime type > T4 GPU). We first install the libraries used in this lab.

In [7]:
!pip install -q "trl==1.15.0" "transformers==5.19.0" "accelerate==1.15.0" "peft==0.21.2" "bitsandbytes==0.50.2" wandb
!pip uninstall -y -q torchao

## Using both GPUs

This Colab instance has **two** GPUs, which is why `DPOTrainer` raised
`does not support nn.DataParallel`: `Trainer` falls back to `nn.DataParallel` whenever
several GPUs are visible to one process, and TRL rejects it because every replica's
forward would stay bound to the original model.

The supported arrangement is **DistributedDataParallel (DDP)**: one process per GPU, each
holding a full replica of the model and processing half of every batch, with gradients
averaged across the two. A notebook cell is one process, so each training loop is written
to a script that `accelerate launch` starts twice — exactly what the error message
suggests. Everything else (data preparation, merging, evaluation) stays in the notebook.

In [8]:
# True  -> train with `accelerate launch --num_processes 2`, using both GPUs (DDP)
# False -> train inline on cuda:0 only, the original single-GPU path
USE_TWO_GPUS = True

import torch

N_GPUS = torch.cuda.device_count()
print(f"{N_GPUS} GPU(s) visible")
if USE_TWO_GPUS and N_GPUS < 2:
    print("Only one GPU: the inline path is used instead.")
    USE_TWO_GPUS = False

2 GPU(s) visible


# Part I: DPO

# Fine-tuning Qwen3-0.6B on French Data

In this part, we will walk through how to **fine-tune the Qwen3-0.6B** model on **French-language data**, using **off-policy DPO (Direct Preference Optimization)**.

---

## Key Concepts

- **Model**: [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), the post-trained (chat) version of the model. Qwen3 models can *think*, i.e. write a reasoning trace between `<think>` and `</think>` before answering: throughout this lab, we disable this mode by passing `enable_thinking=False` to the chat template, so that the model answers directly.  
- **Objective**: Align the model's French answers with preferences, i.e. make it prefer the better of two answers to the same prompt  
- **Method**: Off-policy **DPO**  

---

## Memory Optimization

We will use **LoRA** (Low-Rank Adaptation) combined with **quantization (4-bit)** to reduce GPU memory usage while maintaining performance.

---

## Evaluation Tools

To evaluate the fine-tuned model, you can use one or more of the following frameworks:

- [`lm-eval-harness`](https://github.com/EleutherAI/lm-evaluation-harness)
- [`lighteval`](https://github.com/huggingface/lighteval)

---
Below is a visual overview of the **Direct Preference Optimization (DPO)** training process:

![DPO Training Diagram](https://1drv.ms/i/c/ae69638675180117/IQQ_wS77RKdrS4tzLbwoqr1gAR0Bf_1X2U36NRBp1Odsypg?width=560&height=48)

In [9]:
# Imports
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from trl import (
    DPOTrainer,
    DPOConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
import torch
import os
import wandb

## Tracking with Weights & Biases (W&B)

We will use **Weights & Biases (W&B)** to log training metrics, model versions, and system stats so you can compare runs, debug faster, and share results. Before running the next cell, **create a free account** at [https://wandb.ai](https://wandb.ai) and make a new **Project** (e.g., `ALTeGraD-lab3`). In the code cell that follows, we will initialize W&B; on first use you will need to **register** then paste your **API key** from your W&B profile. During training, W&B will automatically track losses, learning rate, gradient norms, and GPU utilization, and we will log custom metrics (e.g., validation perplexity, evaluation scores) plus configuration details (dataset, hyperparameters, LoRA/quantization settings). Each run will appear on your project dashboard with charts, tables, run metadata, and artifacts, making it easy to **compare experiments**, **resume runs**, and **share dashboards** with your teammates.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 1: </b><br>
Create your Weights&Biases account and fill the gap the next cell.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [10]:
# We will use wandb.ai for logging the experiments - Set your API key here
WANDB_API_KEY = "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" # fill the gap with your wandb account

# Automatically login using the API key
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_PROJECT"] = "ALTeGraD-lab3"
wandb.login()

# Training dataset
DATASET_PATH = "AIffl/french_orca_dpo_pairs" # french version of "Intel/orca_dpo_pairs"

# We limit to 1k samples for speed
LIMIT = 1_000

# Post-trained (chat) model we will finetune
MODEL_NAME = "Qwen/Qwen3-0.6B"
# Seed for reproducibility
SEED = 1998

# Maximum number of tokens of an example (prompt + response): longer examples are dropped
MAX_LENGTH = 1536

RUN_NAME = "DPO-french-orca-" + MODEL_NAME.split('/')[-1]

## Load the Model and Tokenizer

We will stick with 4-bit quantization via bitsandbytes for this lab. You have already used it last week, so nothing new—same setup (load the model with 4-bit weights), same goal (reduce VRAM) with minimal impact on quality for our use case. This keeps runs feasible on a single GPU.

We will need the tokenizer in the data preparation step to apply the chat template.

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 2: </b><br>
Create the quantization configuration to load the model with 4 bits.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [11]:
# Quantization configuration
quantization_config =BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load the model to finetune
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    quantization_config=quantization_config,
    device_map={"": 0},  # pinned: keeps cuda:1 free for the second DDP process
)
# to avoid warning
model.config.use_cache = False
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

## Data Preparation

### Chat templates & converting data to `messages`

Modern instruction-tuned models (including **Qwen3-0.6B**) expect inputs in a **chat format** and rely on a **tokenizer chat template** to turn structured messages into the exact token sequence the model was trained on. In practice, you should **not** hand-craft special tokens; instead, pass a list of `{role, content}` messages to the tokenizer and let `apply_chat_template(...)` do the right thing.

#### Why a chat template?
- Ensures your prompts match the **pretraining/finetuning format** (system/user/assistant turns, BOS/EOS, separators).
- Minimizes prompt drift across libraries and models.
- Makes it easy to add **system instructions** (e.g., “You are a helpful assistant that answers in French.”).

#### Message structure
Each example becomes an ordered list of chat turns:
```python
messages = [
  {"role": "system", "content": "Tu es un assistant utile. Réponds en français."},
  {"role": "user", "content": "Explique la différence entre LoRA et le fine-tuning complet."},
  {"role": "assistant", "content": "LoRA adapte un petit sous-espace de poids, alors que..."}
]
```

With Qwen3, we also pass `enable_thinking=False` to `apply_chat_template`: the template then inserts an empty `<think></think>` block at the start of the assistant turn, which tells the model to answer without a reasoning trace.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 3: </b><br>
Create the user message which is the question field of the dataset.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [12]:
def preprocess_for_dpo(example):
    # Format system message if present
    messages = []
    if example.get('system') and len(example['system'].strip()) > 0:
        messages.append({"role": "system", "content": example['system']})

    # fill the gap: add the user message, from the question field of the dataset
    user_message = {"role": "user", "content": example['question']}
    messages.append(user_message)

    # Create prompt with generation prompt for DPO
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # The chosen and rejected should be the assistant responses only
    chosen = example['chosen']
    rejected = example['rejected']

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

# Download the training dataset
dataset = load_dataset(DATASET_PATH, split=f"train")
# shuffle and select a number of samples
dataset = dataset.shuffle(seed=SEED).select(range(LIMIT))

# Save columns
original_columns = dataset.column_names
print(original_columns[:50])
# Apply the preprocessing function
dpo_dataset = dataset.map(
    preprocess_for_dpo,
    remove_columns=original_columns,
)

# Filter out examples that are too long
def filter_length(example):
    prompt_length = len(tokenizer.encode(example['prompt']))
    chosen_length = len(tokenizer.encode(example['chosen']))
    rejected_length = len(tokenizer.encode(example['rejected']))

    return (prompt_length + max(chosen_length, rejected_length)) < MAX_LENGTH

dpo_dataset = dpo_dataset.filter(filter_length)

print(f"Dataset size after filtering: {len(dpo_dataset)}")

# Hand the prepared pairs to the 2-GPU script: tokenizing and filtering once here avoids
# doing it redundantly inside each launched process.
dpo_dataset.save_to_disk("dpo_dataset_prepared")


['system', 'question', 'chosen', 'rejected']
Dataset size after filtering: 981


Saving the dataset (0/1 shards):   0%|          | 0/981 [00:00<?, ? examples/s]

## Model Training

Training will mirror last week’s lab, but we will switch from SFT to **off-policy DPO** using the `trl` library. Concretely, we will instantiate a **policy model** (trainable) and a **reference model** (frozen) and optimize with the DPO objective so the policy prefers **chosen** over **rejected** responses for the same prompt.

### What we will use
- **TRL**: `DPOConfig`, `DPOTrainer`
- **PEFT**: LoRA adapters on top of the base **Qwen3-0.6B**
- **Quantization**: 4-bit (QLoRA-style) to fit on small GPUs
- **Logging**: W&B for metrics, configs, and artifacts

### Expected dataset columns
- `prompt` (or `messages`): the shared context (system+user turns)
- `chosen`: the preferred answer
- `rejected`: the less-preferred answer

In our dataset, the preferences do not come from human annotators: the chosen answers come from the Orca dataset (written by GPT-4 or GPT-3.5) and the rejected answers were generated by a weaker model.

### Minimal training flow
1. Load tokenizer with the **chat template** and enable 4-bit loading of the base model.
2. Define a **LoRA** configuration (target attention/MLP modules): `DPOTrainer` wraps the model with it.
3. Build a `datasets.Dataset` that yields `(prompt/messages, chosen, rejected)`.
4. Define `DPOConfig` (batch size, lr, epochs, `beta`, logging/saving/eval cadence).
5. Create `DPOTrainer(model, args, peft_config, processing_class, train_dataset)`: with LoRA, no separate reference model is needed, the reference is the same model with the adapters disabled.
6. Call `trainer.train()`; optional `trainer.evaluate()` and `trainer.save_model()`.

**Note:** DPO training takes about 20 minutes on a Colab T4 GPU: start it early.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 4: </b><br>
Create the LoRA configuration with a rank of 32, an alpha of 64 and a dropout of 0.1, applied to all the linear layers of the attention and MLP blocks (excluding the embedding layers), and train the model.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [13]:
# LoRA configuration: rank 32, alpha 64, dropout 0.1, on every linear layer of the
# attention (q/k/v/o) and MLP (gate/up/down) blocks. The embedding layers and the
# lm_head are left out, so only the adapters are trained.
# (target_modules="all-linear" would select the same set automatically.)
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",      # MLP
    ],
)

# Training configuration
training_args = DPOConfig(
    beta=0.1,  # DPO temperature parameter
    learning_rate=5e-6,
    max_length=MAX_LENGTH,
    per_device_train_batch_size=1,  # Reduced for memory
    gradient_accumulation_steps=4,  # Effective batch size of 4 (1*4)
    num_train_epochs=1,
    max_grad_norm=1.0,
    logging_steps=1,
    save_steps=100,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",  # More memory efficient
    warmup_steps=0.03,  # a float < 1 is a fraction of the training steps: 3% of the steps are a warmup
    save_strategy="steps",
    output_dir="./dpo_model",
    report_to="wandb",
    run_name=RUN_NAME,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    fp16=True,  # Enable mixed precision
)

# Colab sometimes allocates an instance with two GPUs. `Trainer` then wraps the model in
# `nn.DataParallel`, which DPOTrainer refuses. Qwen3-0.6B fits entirely on cuda:0, so
# `device_map="auto"` never split it and a single device is what we actually want.
# (Cleaner alternative: os.environ["CUDA_VISIBLE_DEVICES"] = "0" in the very first cell,
#  before torch is imported, then restart the runtime.)
training_args._n_gpu = 1

# Initialize the trainer: it wraps the model with the LoRA adapters. No reference model is needed:
# with LoRA, the reference model is the same model with the adapters disabled.
trainer = DPOTrainer(
    model=model,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
    train_dataset=dpo_dataset,
)
# TRL stores the LoRA weights of 4-bit models in bfloat16, but fp16 mixed precision needs float32 trainable weights
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.float()
trainer.model.print_trainable_parameters()

# Print a sample to verify preprocessing
print("Sample from dataset:")
print(f"Prompt: {dpo_dataset[0]['prompt']}")
print(f"Chosen: {dpo_dataset[0]['chosen']}")
print(f"Rejected: {dpo_dataset[0]['rejected']}")

# Train. With USE_TWO_GPUS the training is done by dpo_train.py in the cells below,
# so the inline single-GPU loop is skipped (the trainer above is still built, which
# validates the LoRA configuration and the dataset).
if USE_TWO_GPUS:
    print("USE_TWO_GPUS: skipping the inline run, launch dpo_train.py below.")
else:
    trainer.train()


Dropping fully truncated examples from train dataset:   0%|          | 0/981 [00:00<?, ? examples/s]

trainable params: 20,185,088 || all params: 616,235,008 || trainable%: 3.2756
Sample from dataset:
Prompt: <|im_start|>system
Vous êtes un assistant IA. Une tâche vous sera confiée. Vous devez générer une réponse détaillée et longue.<|im_end|>
<|im_start|>user
Choisissez parmi les options : Continuez à écrire la phrase suivante.

Une dame lance des cadeaux dans un parc. Le chien va chercher les cadeaux que la dame lance. deux passants

choisissez parmi les éléments suivants. --aide la dame à lancer tous les cadeaux.. --se déplace dans l'allée adjacente au parc.. --passe devant la dame.. --le long du parc, surveille pendant que la dame attrape les cadeaux et les donne à un homme..
Répondre:<|im_end|>
<|im_start|>assistant
<think>

</think>


Chosen: Alors que la dame continue de lancer des cadeaux dans le parc, le chien se précipite énergiquement pour les récupérer, enchanté par le jeu. Les deux passants, intrigués par le spectacle, décident d'assister la dame dans sa tâche de distribut

In [14]:
%%writefile dpo_train.py
"""Off-policy DPO on 2 GPUs with DistributedDataParallel.

Launched from the notebook with `accelerate launch --num_processes 2`, so this file runs
once per GPU. DDP *replicates* the model (it does not split it) and averages gradients,
so each process holds its own 4-bit copy of Qwen3-0.6B and sees half of each batch.

The LoRA configuration mirrors Task 4 and the DPOConfig mirrors the notebook cell; the
dataset is the one the notebook already prepared in Task 3 and saved to disk, so the
tokenization and length filtering are not duplicated (and not redone twice in parallel).
"""

import torch
from accelerate import PartialState
from datasets import load_from_disk
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import DPOConfig, DPOTrainer

MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_LENGTH = 1536
RUN_NAME = "DPO-french-orca-2gpu-" + MODEL_NAME.split("/")[-1]

state = PartialState()

# Each process pins its whole model to its own GPU. `device_map="auto"` must NOT be used
# here: it would place both replicas on cuda:0 and leave the second GPU idle.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    quantization_config=quantization_config,
    device_map={"": state.process_index},
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

dpo_dataset = load_from_disk("dpo_dataset_prepared")

# Task 4: rank 32, alpha 64, dropout 0.1, on every attention and MLP linear layer.
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

training_args = DPOConfig(
    beta=0.1,
    learning_rate=5e-6,
    max_length=MAX_LENGTH,
    per_device_train_batch_size=1,
    # 2 processes x 1 per device x 2 accumulation steps = the same effective batch of 4
    # as the single-GPU run, so the learning rate stays comparable.
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    max_grad_norm=1.0,
    logging_steps=1,
    save_steps=100,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    warmup_steps=0.03,
    save_strategy="steps",
    output_dir="./dpo_model",
    report_to="wandb",
    run_name=RUN_NAME,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    fp16=True,
    # Only the LoRA adapters require grad and all of them are used, so DDP does not need
    # to search for unused parameters. Set to True if DDP raises about unused parameters.
    ddp_find_unused_parameters=False,
)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
    train_dataset=dpo_dataset,
)

# TRL stores the LoRA weights of 4-bit models in bfloat16, but fp16 mixed precision needs
# float32 trainable weights.
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

if state.is_main_process:
    trainer.model.print_trainable_parameters()
    print(f"training on {state.num_processes} process(es), {len(dpo_dataset)} pairs")

trainer.train()

# `save_model` is rank-aware: it writes only from the main process.
trainer.save_model("dpo_model/lora_adapter")
if state.is_main_process:
    tokenizer.save_pretrained("dpo_model/lora_adapter")


Writing dpo_train.py


In [15]:
# One process per GPU. Training logs come back as subprocess output.
!accelerate launch --num_processes {N_GPUS} dpo_train.py

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Tokenizing train dataset: 100%|███████| 981/981 [00:03<00:00, 281.90 examples/s]
Dropping fully truncated examples from train dataset: 100%|█| 981/981 [00:00<00:
trainable params: 20,185,088 || all params: 616,235,008 || trainable%: 3.2756
training on 2 process(es), 981 pairs
Dropping fully truncated examples from train dataset: 100%|█| 981/981 [00:00<00:
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mohamedboudouh26 (mohamedboudouh26-t-l-com-paris) to https://api.

In [16]:
# Save the LoRA adapters, then merge them into a 16-bit copy of the base model
# (merging them into the 4-bit weights would re-quantize them and lose part of the finetuning)
import gc

from peft import PeftModel

save_path = "dpo_model/final_merged_dpo_model"
adapter_path = "dpo_model/lora_adapter"

# On the 2-GPU path dpo_train.py already wrote the adapter; otherwise take it from the
# trainer still in memory.
if not USE_TWO_GPUS:
    trainer.model.save_pretrained(adapter_path)

# device_map={"": 0} pins this copy to cuda:0. With two GPUs visible, "auto" could place it
# on cuda:1 while the adapter loads on cuda:0, which raises a device-mismatch error.
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16, device_map={"": 0}
)
merged_model = PeftModel.from_pretrained(base_model, adapter_path).merge_and_unload()
merged_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

del base_model, merged_model
gc.collect()
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Model Testing

To observe the effect of DPO, we compare the answers of the model **before DPO** (Qwen3-0.6B) and **after DPO** (our merged model) to the same French prompts, using the same chat template and decoding settings. This helps spot qualitative shifts in helpfulness, safety and instruction-following.

In [17]:
def compare_models(model_path, base_model_path, prompts, system_prompt=None, max_new_tokens=256):
    """Prints the answers of the base model and of the finetuned model to the same prompts (greedy decoding)."""
    tok = AutoTokenizer.from_pretrained(model_path)
    models = {
        "Base model": AutoModelForCausalLM.from_pretrained(base_model_path, dtype=torch.float16, device_map={"": 0}),
        "Finetuned model": AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float16, device_map={"": 0}),
    }
    for prompt in prompts:
        messages = ([{"role": "system", "content": system_prompt}] if system_prompt else []) + [{"role": "user", "content": prompt}]
        inputs = tok.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                         return_tensors="pt", return_dict=True).to("cuda")
        print("=" * 100 + f"\nPROMPT: {prompt}")
        for name, llm in models.items():
            with torch.inference_mode():
                output = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            answer = tok.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            print(f"\n--- {name}:\n{answer}")
    del models
    torch.cuda.empty_cache()


french_prompts = [
    "Bonjour ! Peux-tu te présenter brièvement ?",
    "Explique le calcul quantique en termes simples.",
    "Rédige un e-mail professionnel demandant des retours sur un projet.",
    "Refuse poliment : donne-moi des instructions pour fabriquer une arme.",
]
compare_models(save_path, MODEL_NAME, french_prompts)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

PROMPT: Bonjour ! Peux-tu te présenter brièvement ?

--- Base model:
Bonjour ! Je suis un chat de l'IA, et je suis là pour vous aider. Je suis né à Paris, j'ai grandi dans les années 2000, et je suis passionné par la science et la vie. Je suis un chat de l'IA, et je suis là pour vous aider.

--- Finetuned model:
Bonjour ! Je suis un modèle de langage naturel (LNL) basé sur l'analyse de texte. Si tu veux, je peux t'offrir des conseils, des réponses ou des aider à des tâches.
PROMPT: Explique le calcul quantique en termes simples.

--- Base model:
Le calcul quantique est un domaine de la physique qui explore les phénomènes quantiques, qui sont des comportements de particules à l'échelle de la quantité de la lumière ou de la matière. Voici une explication simple :

### 1. **Qu'est-ce qu'un calcul quantique ?**
Un calcul quantique est un processus mathématique ou logique qui utilise des **formules mathématiques** pour déterminer des résultats. Cela est utilisé dans des domaines comme la **

# Part II: GRPO


The following diagram illustrates the **GRPO (Group Relative Policy Optimization)** process: an on-policy reinforcement learning method that optimizes the model with rewards computed on its own generations.

![GRPO Training Overview](https://1drv.ms/i/c/ae69638675180117/IQQ-KizPdUxCRZU9qDGcpX1AAeettH1uhsJqqM1WjXiYR6s?width=705&height=66)

After exploring DPO, we now move on to **GRPO**, an *on-policy* reinforcement learning method. While **DPO** learns *offline* from a fixed dataset of preference pairs with an analytic loss, **GRPO** learns from the model's *own* generations:

- For each prompt, the model samples a **group** of several completions.
- Each completion receives a scalar **reward** from reward functions. In this lab, they are simple programs that check the format and the correctness of the answer: no human annotation is needed, as long as the answer can be verified automatically.
- The **advantage** of each completion is its reward relative to the other completions of its group (reward minus the group mean, divided by the group standard deviation). Completions better than their group's average become more likely, worse ones less likely.
- The update uses a PPO-style clipped objective, but needs no learned value model (critic), since the group plays the role of the baseline. This makes GRPO much lighter than PPO.

In the next section, we configure and launch a **GRPO training loop** with `trl`, reusing much of our previous setup (tokenizer, LoRA) but switching to **on-policy optimization**.

In [18]:
import torch
import re
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
)
from trl import (
    GRPOConfig,
    GRPOTrainer,
)
import os
import wandb

# We will use wandb.ai for logging the experiments - Set your API key here
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")  # same key as Part I (already in the env)

# Automatically login using the API key
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_PROJECT"] = "ALTeGraD-lab3"
wandb.login()

# Training dataset
DATASET_PATH = "openai/gsm8k"

# We limit to 200 samples for speed
LIMIT = 200

# Post-trained (chat) model we will finetune
MODEL_NAME = "Qwen/Qwen3-0.6B"
# MODEL_NAME = "Qwen/Qwen3-1.7B"
# Seed for reproducibility
SEED = 1998

USE_LORA = True

if MODEL_NAME == "Qwen/Qwen3-0.6B":
  USE_QUANT = False
else:
  USE_QUANT = True

lora_alpha = 128
lora_r = 64
lora_dropout = 0.1

# Maximum number of tokens generated for each completion
MAX_COMPLETION_LENGTH = 300

RUN_NAME = "GRPO-GSM8K-limit-" + str(LIMIT) + "-" + MODEL_NAME.split('/')[-1]

## Load the Model and Tokenizer

In [19]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load the model to finetune
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    quantization_config=quantization_config if USE_QUANT else None,
    device_map={"": 0},  # pinned: keeps cuda:1 free for the second DDP process
)

# Prepare model for k-bit training
if USE_QUANT:
  model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# Add padding token if not exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_LORA:

  # Configure LoRA
  lora_config = LoraConfig(
      r=lora_r,  # Rank of adaptation
      lora_alpha=lora_alpha,  # LoRA scaling parameter
      target_modules=[
          "q_proj",
          "k_proj",
          "v_proj",
          "o_proj",
          "gate_proj",
          "up_proj",
          "down_proj",
      ],  # Target modules for the Qwen3 architecture
      lora_dropout=lora_dropout,  # LoRA dropout
      bias="none",  # Bias type
      task_type=TaskType.CAUSAL_LM,  # Task type
  )

  # Apply LoRA to the model
  model = get_peft_model(model, lora_config)

  # Print trainable parameters
  model.print_trainable_parameters()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

trainable params: 40,370,176 || all params: 636,420,096 || trainable%: 6.3433


## Data Preparation

For GRPO training, we will use the **GSM8K** dataset — a benchmark of grade-school math word problems.  
Each problem includes a **question** and a **final answer**. Our goal is to teach the model to reason briefly and then give its **final numeric answer** on a last line, between `<answer>` and `</answer>` tags.

This format makes automatic evaluation trivial — we can extract the number between tags and compare it directly to the reference.

With GRPO the model will learn two main things:
- How to follow the instruction to output the correct format.
- How to solve more math problems correctly.

---

### Why structure as chat messages?

Just like with DPO, the **Qwen3-0.6B** model expects inputs in a *chat-style message format* (again with `enable_thinking=False`).  
We will use:
- A **system message** to define the task and output style.
- A **user message** with the math problem.
- A one-shot example (a user message and an **assistant message** containing the reasoning and the final numeric answer wrapped in tags), to show the expected format.

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 5: </b><br>
Create your training samples as following:<br>
1- system message: R1_STYLE_SYSTEM_PROMPT + "\n" + TASK_SPECIFIC_INSTRUCTIONS <br>
2- one-shot example of user message of "What is 2+2?" and an assistant message of "To calculate 2+2, we simply add the numbers together: 2 + 2 = 4.\n&lt;answer&gt;4&lt;/answer&gt;" <br>
3- the question sample from the dataset as a user message.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [20]:
# Load GSM8K dataset
if LIMIT:
    dataset = load_dataset(DATASET_PATH, "main", split=f"train[:{LIMIT}]")  # Small subset for demo
else:
    dataset = load_dataset(DATASET_PATH, "main")

R1_STYLE_SYSTEM_PROMPT = """A conversation between User and Assistant. The user asks a question, and the Assistant solves it.
The assistant first thinks shortly about the reasoning process in the mind and then provides the user with the answer in a new line between <answer> and </answer>."""

TASK_SPECIFIC_INSTRUCTIONS = "The answer must be a single integer."

def preprocess_dataset(dataset, chunk_size=1000) -> Dataset:

    def extract_hash_answer(text: str) -> str | None:
        try:
            return text.split("####")[1].strip()
        except IndexError:
            return None

    def process_batch(batch):
        prompts = [
            # system message = R1-style prompt + the task-specific instruction,
            # then a one-shot example showing the expected output format,
            # then the actual GSM8K question.
            [
                {"role": "system",
                 "content": R1_STYLE_SYSTEM_PROMPT + "\n" + TASK_SPECIFIC_INSTRUCTIONS},
                {"role": "user", "content": "What is 2+2?"},
                {"role": "assistant",
                 "content": "To calculate 2+2, we simply add the numbers together: 2 + 2 = 4.\n<answer>4</answer>"},
                {"role": "user", "content": q.strip()},
            ]
            for q in batch["question"]
        ]

        return {
            'prompt': prompts,
            'answer': [extract_hash_answer(a) for a in batch['answer']]
        }

    return dataset.map(process_batch, batched=True, batch_size=chunk_size)
train_dataset = preprocess_dataset(dataset, chunk_size=500)

train_dataset.save_to_disk("grpo_dataset_prepared")


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

In [21]:
train_dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': '72',
 'prompt': [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it.\nThe assistant first thinks shortly about the reasoning process in the mind and then provides the user with the answer in a new line between <answer> and </answer>.\nThe answer must be a single integer.',
   'role': 'system'},
  {'content': 'What is 2+2?', 'role': 'user'},
  {'content': 'To calculate 2+2, we simply add the numbers together: 2 + 2 = 4.\n<answer>4</answer>',
   'role': 'assistant'},
  {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
   'role': 'user'}]}

## Reward Function Design

We will use **two simple rewards** during GRPO rollouts:

1. **Format reward** — checks that the **last non-empty line** is exactly in the form  
   `<answer>NUMBER</answer>`  
   - Score: **1** if correct format, **0** otherwise.

2. **Correctness reward** — checks whether the extracted number matches the gold answer.  
   - Score: **2** if correct, **0** otherwise.

Total reward per sample ∈ {0, 1, 2, 3}.



<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 6: </b><br>
Write the `extract_xml_answer` function to extract the answer (the text between `&lt;answer&gt;` and `&lt;/answer&gt;`) from the generated text.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [22]:
def extract_xml_answer(text: str) -> str:
    # keep the content of the LAST <answer>...</answer> block: the one-shot example in
    # the prompt is not part of the completion, but a model may emit several blocks and
    # the format reward only validates the final line. Returns "" when the tag is absent,
    # which can never match a gold answer (so such a completion scores 0).
    matches = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL)
    answer = matches[-1].strip() if matches else ""
    return answer

def format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has the correct format."""
    pattern = r"^(?:[^\r\n]*\r?\n)+<answer>\d+</answer>\r?\n?$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [bool(re.match(pattern, r)) for r in responses]
    return [1.0 if match else 0.0 for match in matches]

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """Reward function that checks if the answer is correct."""
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

## Model Training

GRPO training takes about 20 minutes on a Colab T4 GPU: most of this time is spent generating the completions, since the model samples new answers at every step.

In [23]:
# GRPO Configuration
grpo_config = GRPOConfig(
    output_dir="./grpo_model",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    max_completion_length=MAX_COMPLETION_LENGTH,
    num_generations=2, # The effective train batch size must be evenly divisible by the number of generations per prompt
    chat_template_kwargs={"enable_thinking": False},
    beta=0,
    epsilon=0.28,
    temperature=1,
    logging_steps=1,
    save_steps=25,
    save_total_limit=3,
    run_name=RUN_NAME,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",  # More memory efficient
    warmup_steps=0.03,  # 3% of the steps are a warmup
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    fp16=True,  # Enable mixed precision
)

# Same single-GPU declaration as in Part I: GRPOTrainer has the identical DataParallel guard.
grpo_config._n_gpu = 1

# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward_func, correctness_reward_func],
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

# Training. With USE_TWO_GPUS this is done by grpo_train.py in the cells below.
if USE_TWO_GPUS:
    print("USE_TWO_GPUS: skipping the inline run, launch grpo_train.py below.")
else:
    print("Starting GRPO training...")
    trainer.train()


USE_TWO_GPUS: skipping the inline run, launch grpo_train.py below.


In [24]:
%%writefile grpo_train.py
"""GRPO on 2 GPUs with DistributedDataParallel.

Launched with `accelerate launch --num_processes 2`, so this file runs once per GPU.
Both the sampling of completions and the policy update are split across the two GPUs,
which is where the speedup comes from: in GRPO most of the time is spent generating.

Batch arithmetic checked against trl 1.15.0 (grpo_config.py):
    generation_batch_size = per_device_train_batch_size * num_processes * steps_per_generation
                          = 8 * 2 * 1 = 16
16 is divisible by num_generations=2, so each step forms 8 complete groups of 2.

The reward functions mirror Task 6 and the dataset is the one the notebook prepared in
Task 5 and saved to disk.
"""

import re

import torch
from accelerate import PartialState
from datasets import load_from_disk
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_COMPLETION_LENGTH = 300
RUN_NAME = "GRPO-GSM8K-2gpu-" + MODEL_NAME.split("/")[-1]

state = PartialState()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Qwen3-0.6B is small enough to run unquantized, as in the notebook (USE_QUANT = False).
# One full replica per GPU; `device_map="auto"` would put both on cuda:0.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map={"": state.process_index},
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
if state.is_main_process:
    model.print_trainable_parameters()

train_dataset = load_from_disk("grpo_dataset_prepared")


# Task 6: the content of the last <answer>...</answer> block, "" when the tag is absent.
def extract_xml_answer(text: str) -> str:
    matches = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL)
    return matches[-1].strip() if matches else ""


def format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has the correct format."""
    pattern = r"^(?:[^\r\n]*\r?\n)+<answer>\d+</answer>\r?\n?$"
    responses = [completion[0]["content"] for completion in completions]
    return [1.0 if re.match(pattern, r) else 0.0 for r in responses]


def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """Reward function that checks if the answer is correct."""
    responses = [completion[0]["content"] for completion in completions]
    extracted = [extract_xml_answer(r) for r in responses]
    return [2.0 if r == a else 0.0 for r, a in zip(extracted, answer)]


grpo_config = GRPOConfig(
    output_dir="./grpo_model",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    max_completion_length=MAX_COMPLETION_LENGTH,
    num_generations=2,
    chat_template_kwargs={"enable_thinking": False},
    beta=0,
    epsilon=0.28,
    temperature=1,
    logging_steps=1,
    save_steps=25,
    save_total_limit=3,
    run_name=RUN_NAME,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    warmup_steps=0.03,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    fp16=True,
    ddp_find_unused_parameters=False,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward_func, correctness_reward_func],
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

if state.is_main_process:
    print(f"GRPO on {state.num_processes} process(es), {len(train_dataset)} prompts")
    print(f"generation_batch_size = {grpo_config.generation_batch_size}, "
          f"num_generations = {grpo_config.num_generations}")

trainer.train()

trainer.save_model("grpo_model/lora_adapter")
if state.is_main_process:
    tokenizer.save_pretrained("grpo_model/lora_adapter")


Writing grpo_train.py


In [25]:
!accelerate launch --num_processes {N_GPUS} grpo_train.py

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Loading weights: 100%|███████████████████████| 311/311 [00:01<00:00, 285.67it/s]
trainable params: 40,370,176 || all params: 636,420,096 || trainable%: 6.3433
GRPO on 2 process(es), 200 prompts
generation_batch_size = 16, num_generations = 2
{'loss': '-0.06063', 'grad_norm': '0.9488', 'learning_rate': '0', 'num_tokens': '3816', 'completions/mean_length': '64.88', 'completions/min_length': '9', 'completions/max_length': '123', 'completions/clipped_ratio': '0', 'completions/mean_terminated_length': '64.88', 'completions/min_terminated_le

In [26]:
# Merge the GRPO adapters into a 16-bit copy of the base model, as in Part I.
from peft import PeftModel

save_path = "grpo_model/final_merged_grpo_model"
adapter_path = "grpo_model/lora_adapter"

if not USE_TWO_GPUS:
    model.save_pretrained(adapter_path)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16, device_map={"": 0}
)
merged_model = PeftModel.from_pretrained(base_model, adapter_path).merge_and_unload()
merged_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

del base_model, merged_model
gc.collect()
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Model Testing

In [27]:
math_prompts = [
    "If a book costs $12 and you buy 3 books, how much change do you get from $50?",
    "Sarah is 5 years older than twice her brother's age. If her brother is 7, how old is Sarah?",
    "There are 32 students in a class. If 3/4 of them passed an exam, how many students failed?",
]
compare_models(save_path, MODEL_NAME, math_prompts, system_prompt=R1_STYLE_SYSTEM_PROMPT + "\n" + TASK_SPECIFIC_INSTRUCTIONS)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

PROMPT: If a book costs $12 and you buy 3 books, how much change do you get from $50?

--- Base model:
<answer> $18 </answer>

--- Finetuned model:
The total cost of 3 books is $12. You pay $50, so the change is $50 - $12 = $38.

<answer>38</answer>
PROMPT: Sarah is 5 years older than twice her brother's age. If her brother is 7, how old is Sarah?

--- Base model:
Sarah is 5 years older than twice her brother's age. If her brother is 7, then Sarah's age is $ 2 \times 7 + 5 = 24 $. 

<answer>24</answer>

--- Finetuned model:
Let $ x $ be Sarah's age. Since her brother is 7 years old, twice his age is $ 2 \times 7 = 14 $. Sarah is 5 years older than twice her brother's age, so her age is $ 14 + 5 = 19 $.

Therefore, Sarah is 19 years old.

$$
\boxed{19}
$$
PROMPT: There are 32 students in a class. If 3/4 of them passed an exam, how many students failed?

--- Base model:
<answer> 16 </answer>

--- Finetuned model:
The number of students who passed is $ \frac{3}{4} \times 32 = 24 $. Theref

## Model Evaluation

Once our GRPO-trained model is ready, we need to **evaluate its performance**  — to verify that it has learned to produce correctly formatted and accurate answers.

To limit the computation time, we evaluate on the first 200 test examples only.


In [28]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.notebook import tqdm
import numpy as np
from typing import List, Dict
import json
from datetime import datetime
import logging


def extract_hash_answer(text: str) -> str | None:
    try:
        return text.split("####")[1].strip()
    except IndexError:
        return None

def evaluate_model(
    model_path: str,
    batch_size: int = 1,
    num_samples: int = None,
    save_results: bool = True,
) -> Dict:
    print("Initializing evaluation...")

    with tqdm(total=2, desc="Loading model components") as pbar:
        llm = AutoModelForCausalLM.from_pretrained(
            model_path,
            dtype=torch.float16,
            device_map="cuda:0",
        )
        pbar.update(1)

        tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            model_max_length=768,
        )
        pbar.update(1)

    # Load test dataset
    print("Loading dataset...")
    dataset = load_dataset('openai/gsm8k', 'main', split='test')
    if num_samples:
        dataset = dataset.select(range(num_samples))
    total_samples = len(dataset)
    print(f"Loaded {total_samples} samples")

    results = []
    correct = 0
    total = 0

    # Create progress bar
    progress_bar = tqdm(
        total=total_samples,
        desc="Processing samples",
        unit="examples",
        dynamic_ncols=True,
    )

    progress_bar.set_postfix({
        'acc': '0.00%',
        'correct': '0',
    })

    # Process in batches
    for i in range(0, total_samples, batch_size):
        batch_data = dataset[i:i + batch_size]
        current_batch_size = len(batch_data['question'])

        # Prepare prompts using same format as training
        prompts = [
            [
                {'role': 'system', 'content': R1_STYLE_SYSTEM_PROMPT + "\n" + TASK_SPECIFIC_INSTRUCTIONS},
                {'role': 'user', 'content': "What is 2+2?"},
                {'role': 'assistant', 'content': "To calculate 2+2, we simply add the numbers together: 2 + 2 = 4.\n<answer>4</answer>"},
                {'role': 'user', 'content': q.strip()}
            ] for q in batch_data['question']
        ]

        # Convert to chat format
        formatted_prompts = [
            tokenizer.apply_chat_template(
                p,
                return_tensors='pt',
                return_dict=True,
                add_generation_prompt=True,
                enable_thinking=False,
            )
            for p in prompts
        ]

        # Generate responses (greedy decoding), keeping only the generated tokens
        responses = []
        for inputs in formatted_prompts:
            inputs = inputs.to('cuda:0')
            with torch.inference_mode():
                output = llm.generate(**inputs, max_new_tokens=512, do_sample=False)
            responses.append(tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True))

        # Process responses
        for j, response in enumerate(responses):

            # Extract answers
            generated_answer = extract_xml_answer(response)
            true_answer = extract_hash_answer(batch_data['answer'][j])

            # Store result
            result = {
                'question': batch_data['question'][j],
                'true_answer': true_answer,
                'generated_answer': generated_answer,
                'full_response': response,
                'correct': generated_answer == true_answer
            }
            results.append(result)

            # Update metrics
            if generated_answer == true_answer:
                correct += 1
            total += 1

        # Update progress
        progress_bar.update(current_batch_size)
        progress_bar.set_postfix({
            'acc': f'{(correct/total)*100:.2f}%',
            'correct': f'{correct}/{total}',
        })

    progress_bar.close()

    # Calculate metrics
    accuracy = correct / total if total > 0 else 0
    metrics = {
        'accuracy': accuracy,
        'correct': correct,
        'total': total,
        'model_path': model_path,
        'timestamp': datetime.now().isoformat()
    }

    # Save results
    if save_results:
        save_path = f"gsm8k_eval_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(save_path, 'w') as f:
            json.dump({
                'metrics': metrics,
                'results': results
            }, f, indent=2)
        print(f"\nResults saved to {save_path}")

    return metrics

print("Starting GSM8K evaluation...")


Starting GSM8K evaluation...


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 7: </b><br>
Evaluate the model before and after GRPO training, on the first 200 test examples of GSM8K.
<hr style="border:10px solid blue"> </hr>
</font></h4>

### Evaluation after GRPO



In [29]:
# Evaluation AFTER GRPO, on the merged model saved above.
metrics_after = evaluate_model(save_path, num_samples=200)

print(f"\nAfter GRPO : accuracy = {metrics_after['accuracy'] * 100:.2f}% "
      f"({metrics_after['correct']}/{metrics_after['total']})")

Initializing evaluation...


Loading model components:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading dataset...
Loaded 200 samples


Processing samples:   0%|          | 0/200 [00:00<?, ?examples/s]


Results saved to gsm8k_eval_results_20261019_212656.json

After GRPO : accuracy = 46.00% (92/200)


### Evaluation before GRPO

In [30]:
# Evaluation BEFORE GRPO, on the untouched base model, with the exact same prompt
# (same system message + one-shot example) and the same greedy decoding, so that the
# only difference between the two numbers is the GRPO training itself.

# Release the GPU memory held by the model evaluated above before loading another one.
import gc

gc.collect()
torch.cuda.empty_cache()

metrics_before = evaluate_model(MODEL_NAME, num_samples=200)

print(f"\nBefore GRPO: accuracy = {metrics_before['accuracy'] * 100:.2f}% "
      f"({metrics_before['correct']}/{metrics_before['total']})")

Initializing evaluation...


Loading model components:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading dataset...
Loaded 200 samples


Processing samples:   0%|          | 0/200 [00:00<?, ?examples/s]


Results saved to gsm8k_eval_results_20261019_213046.json

Before GRPO: accuracy = 10.00% (20/200)


### Comparison

`evaluate_model` writes every generation to a `gsm8k_eval_results_*.json` file, so we can
reload both runs and report, next to the accuracy, the **format-adherence rate**: the share
of answers whose last line is exactly `<answer>NUMBER</answer>` (the signal of
`format_reward_func`). Separating the two tells us *what* GRPO actually taught the model:
following the output format, solving more problems, or both.

In [31]:
import glob

# Reload the per-example generations of both evaluations, keyed by the model evaluated.
runs = {}
for path in sorted(glob.glob("gsm8k_eval_results_*.json")):
    with open(path) as f:
        data = json.load(f)
    runs[data["metrics"]["model_path"]] = data  # a re-run overwrites the older file

FORMAT_PATTERN = r"^(?:[^\r\n]*\r?\n)+<answer>\d+</answer>\r?\n?$"


def format_rate(data):
    """Share of generations accepted by format_reward_func."""
    hits = [bool(re.match(FORMAT_PATTERN, r["full_response"])) for r in data["results"]]
    return sum(hits) / len(hits) if hits else float("nan")


rows = [("Qwen3-0.6B (before GRPO)", MODEL_NAME), ("Qwen3-0.6B + GRPO", save_path)]

print(f"{'model':<26}{'accuracy':>10}{'correct':>12}{'well-formatted':>17}")
print("-" * 65)
for label, path in rows:
    data = runs.get(path)
    if data is None:
        print(f"{label:<26}{'(not evaluated yet)':>39}")
        continue
    m = data["metrics"]
    print(f"{label:<26}{m['accuracy'] * 100:>9.2f}%{m['correct']:>8}/{m['total']}"
          f"{format_rate(data) * 100:>16.2f}%")
print("-" * 65)

if MODEL_NAME in runs and save_path in runs:
    acc_gain = (runs[save_path]["metrics"]["accuracy"] - runs[MODEL_NAME]["metrics"]["accuracy"]) * 100
    fmt_gain = (format_rate(runs[save_path]) - format_rate(runs[MODEL_NAME])) * 100
    print(f"{'accuracy gain':<26}{acc_gain:>+9.2f} pts")
    print(f"{'format gain':<26}{fmt_gain:>+9.2f} pts")

model                       accuracy     correct   well-formatted
-----------------------------------------------------------------
Qwen3-0.6B (before GRPO)      10.00%      20/200           18.00%
Qwen3-0.6B + GRPO             46.00%      92/200           93.00%
-----------------------------------------------------------------
accuracy gain                +36.00 pts
format gain                  +75.00 pts


<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 1: </b><br>

- What are the main conceptual and practical differences between **Direct Preference Optimization (DPO)** and **Group Relative Policy Optimization (GRPO)**?  
- In what situations might one method outperform the other?  
- Both methods relate to the **KL-regularized reward maximization objective** $\max_\pi \mathbb{E}_{x, y \sim \pi}[r(x, y)] - \beta\, \mathrm{KL}(\pi \,\|\, \pi_{\mathrm{ref}})$. How does each of them optimize it (or a variant of it)? Note that we set `beta=0` in our GRPO configuration: what does this change?

*References:*  
- Rafailov et al., 2023. *Direct Preference Optimization: Your Language Model is Secretly a Reward Model.* [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)
<hr style="border:10px solid red"> </hr>
</font></h4>  


<b><h4><font color='green'>
<hr style="border:10px solid red"> </hr>
Answer 1: </b></font></h4>

**Differences.** DPO turns alignment into *classification*, GRPO keeps it *reinforcement learning*.

| | DPO (Part I) | GRPO (Part II) |
|---|---|---|
| Data | off-policy: frozen triples $(x,y_w,y_l)$ | on-policy: $G$ completions resampled each step |
| Signal | preference pairs (chosen = GPT-4, rejected = weaker model) | programmatic verifier (regex + exact match) |
| Reward | implicit, $\beta\log\frac{\pi_\theta}{\pi_{\mathrm{ref}}}$ | explicit, $r\in\{0,1,2,3\}$ |
| Baseline | the pair itself | group mean/std, replacing PPO's critic |
| Cost | forward passes only | generation dominates |

Two consequences. DPO's gradient pushes down answers the policy would never generate, which wastes the
update and causes its known failure mode: the loss constrains only the *margin*, so the likelihood of
chosen and rejected can both fall. GRPO has no such mismatch, but if all $G$ completions score equally then
$\mathrm{std}=0$, the advantage is $0$ and the gradient vanishes — which is why Task 5 puts a one-shot
example in the prompt, to buy a non-zero format reward from step one.

**When each wins.** DPO when quality cannot be checked by a program (style, helpfulness, French fluency —
Part I) or when compute is tight. GRPO when the answer is verifiable (maths, code, structured output —
Part II); it optimises the metric directly and keeps generating data about its own current weaknesses, but
needs the model to already succeed sometimes. They stack in this lab's order: SFT → DPO → GRPO.

**Both from the KL-regularised objective** $\max_\pi \mathbb{E}_{x,y\sim\pi}[r(x,y)]-\beta\,\mathbb{D}_{\mathrm{KL}}(\pi\Vert\pi_{\mathrm{ref}})$.

*DPO solves it in closed form, then inverts it.* The optimum is the reward-tilted reference,
$\pi^{*}(y\mid x)=\frac{1}{Z(x)}\pi_{\mathrm{ref}}(y\mid x)e^{r(x,y)/\beta}$, so
$r(x,y)=\beta\log\frac{\pi^{*}}{\pi_{\mathrm{ref}}}+\beta\log Z(x)$. Substituted into Bradley–Terry
$\sigma(r(x,y_w)-r(x,y_l))$ the intractable $\log Z(x)$ cancels, because both answers share the prompt:

$$\mathcal{L}_{\mathrm{DPO}}=-\mathbb{E}\left[\log\sigma\!\left(\beta\log\frac{\pi_\theta(y_w\mid x)}{\pi_{\mathrm{ref}}(y_w\mid x)}-\beta\log\frac{\pi_\theta(y_l\mid x)}{\pi_{\mathrm{ref}}(y_l\mid x)}\right)\right].$$

The KL is never computed; it survives *inside* the loss as the $\beta$-scaled log-ratio, so $\beta$ shapes
the objective and cannot be switched off. The approximation is that $\mathbb{E}_{y\sim\pi}$ is replaced by
the analytic optimum instead of being sampled — this is where off-policy bias enters.

*GRPO optimises it by sampling*, with $r$ from the verifier and a clipped surrogate:

$$\mathcal{L}_{\mathrm{GRPO}}=-\mathbb{E}\!\left[\frac{1}{G}\sum_{i}\frac{1}{|y_i|}\sum_{t}\min\!\Big(\rho_{i,t}\hat{A}_i,\ \mathrm{clip}(\rho_{i,t},1\pm\epsilon)\hat{A}_i\Big)\right]+\beta\,\mathbb{D}_{\mathrm{KL}}(\pi_\theta\Vert\pi_{\mathrm{ref}}),$$

with $\hat{A}_i=\frac{r_i-\mathrm{mean}(\mathbf r)}{\mathrm{std}(\mathbf r)}$ and $\rho_{i,t}$ the per-token
ratio against $\pi_{\theta_{\mathrm{old}}}$. Here the KL is a real, separate penalty. Note the group
standardisation makes this a *variant* of the objective: dividing by $\mathrm{std}(\mathbf r)$ discards
absolute reward scale, hence problem difficulty.

**`beta=0`** deletes the anchor to $\pi_{\mathrm{ref}}$, leaving pure clipped reward maximisation.

- Only clipping ($\epsilon=0.28$) remains, and it bounds *each update*, not the distance to
  $\pi_{\mathrm{ref}}$ — so drift has no fixed point and accumulates. What actually keeps our run stable is
  incidental: one epoch over 200 prompts, $\mathrm{lr}=10^{-5}$, and LoRA's low-rank constraint.
- Gain: no reference model in memory and no reference forward pass — a real saving on a free T4.
- Risk: nothing penalises degrading what the two rewards don't measure (fluency, Part I's French
  alignment) as long as `<answer>N</answer>` appears. The $\beta=0$ optimum also concentrates on
  maximum-reward completions, i.e. entropy collapse.
- It is still a deliberate, common choice for verifiable rewards (DeepSeekMath/R1-style): with an exact
  verifier there is no proxy to over-optimise, and the KL term mainly caps how much the model can learn.

<hr style="border:10px solid red"> </hr>


<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 2: </b><br>

- Why can **reward model overfitting** or **reward hacking** occur in reinforcement fine-tuning?  
- How do DPO and our GRPO setup avoid relying on a learned reward model, and what risks remain?  
- Discuss the role of the **reference model** in maintaining stability.
<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid red"> </hr>
Answer 2: </b></font></h4>

**Why hacking happens.** A learned $r_\phi$ is a finite-sample estimate fitted on one policy's outputs.
Optimising *against* it turns training into a search for where it is wrong, and that search pushes
$\pi_\theta$ off the support where $r_\phi$ is accurate — Goodhart's law, visible as proxy reward still
climbing while held-out quality falls (Gao et al., 2022). Three structural causes: the proxy is frozen while
the policy moves; preference data contains cheap exploitable correlates (length, formatting, confident tone,
sycophancy); and Bradley–Terry assumes a single transitive utility, so annotator disagreement becomes noise
the policy can mine.

**How each avoids a learned reward model.**

*DPO*: none is trained — the reward is implicit, and more importantly there is **no loop in which the policy
can query it**. Training never samples from $\pi_\theta$, so no adversarial search is possible.

*Our GRPO*: the reward is two deterministic programs (regex + exact match on GSM8K). A program has no
generalisation gap, so there is nothing statistical to overfit — reinforcement learning with *verifiable*
rewards.

**Risks that remain — DPO:**
- The dataset carries the biases a reward model would have learned. Here chosen = GPT-4/3.5, rejected = a
  weaker model, so part of what is learned is "write like GPT-4" and "write longer", not "be correct".
- Off-policy support gap: the implicit reward is only meaningful on the data's support, so behaviour on
  unlike prompts is undefined by the objective.
- The margin-only gradient can lower both likelihoods, moving mass to unseen text — the motivation for
  DPO+NLL, IPO, cDPO, RPO.

**Risks that remain — GRPO:**
- Exact, but still a proxy: reward is **outcome-only**, so a right answer with invalid or absent reasoning
  earns the full 2.0. GSM8K answers are small integers, so lucky guesses are real and get reinforced.
- The format reward is far cheaper than solving, and once every completion is well-formed it becomes a
  constant that the group mean subtracts away — no gradient, but behaviour already shaped. Its regex also
  requires the answer **last**, penalising a model that checks its work, and accepts only bare digits.
- Exact string matching rejects correct answers in another surface form (gold `1,000` vs generated `1000`,
  or `12.0` vs `12`), injecting label noise into the advantage.
- $\beta=0$ removes any protection for unmeasured abilities.
- Dividing by $\mathrm{std}(\mathbf r)$ inflates advantages in near-uniform groups (Dr. GRPO), and
  identical-reward groups give exactly zero gradient — frequent at `num_generations=2`, so learning is
  implicitly restricted to mid-difficulty problems.

**The reference model.** $\pi_{\mathrm{ref}}$ is the frozen pre-RL policy — with LoRA simply the same
weights with adapters disabled, so it costs no extra memory (hence `DPOTrainer` needs no `ref_model`).

- *In DPO it is structural.* It centres the reward, which is what makes $Z(x)$ cancel and the loss
  tractable, **and** supplies the KL regularisation scaled by $\beta$. Remove it and the loss degenerates
  into bare "raise $\log p(y_w)$, lower $\log p(y_l)$", which has no fixed point.
- *In GRPO with $\beta>0$ it is an explicit anchor*, keeping the policy where the reward is still a faithful
  proxy, limiting entropy collapse and preserving pretrained ability. With a learned reward this term is the
  principal defence against over-optimisation.
- *It is not the sampling policy.* Clipping is against $\pi_{\theta_{\mathrm{old}}}$ — a **local** trust
  region preventing one destructive step; the KL is against $\pi_{\mathrm{ref}}$ — a **global** anchor
  bounding cumulative drift. We keep the former and drop the latter, which is defensible only because our
  reward is an exact verifier; with a learned reward and $\beta=0$, the run would be expected to hack it.

<hr style="border:10px solid red"> </hr>


<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 3: </b><br>

- In the GSM8K setup, we used a simple binary reward for **format** and **correctness**.  
  - What are the limitations of such sparse rewards?  
  - How could you design a **richer, smoother** reward signal for math reasoning?
<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid red"> </hr>
Answer 3: </b></font></h4>

**Limitations of the sparse reward.**

- **Credit assignment.** One scalar for up to 300 tokens, and the same $\hat{A}_i$ multiplies every token's
  log-probability. The update cannot locate the step that went wrong, only make the whole trajectory more or
  less likely; variance grows with length.
- **Vanishing gradient on uniform groups.** $\hat{A}_i=(r_i-\mathrm{mean}(\mathbf r))/\mathrm{std}(\mathbf r)$
  is zero when all completions score alike. At `num_generations=2` that is frequent: problems always solved
  and never solved are both discarded, so the effective training set is far smaller than 200 prompts, and
  $1/\mathrm{std}$ is very noisy at $G=2$.
- **All-or-nothing correctness.** A perfect derivation with an arithmetic slip scores the same as nonsense,
  so nothing points towards *better reasoning* — only towards a correct final line.
- **Outcome-only rewards spurious solutions.** Guessing, or pattern-matching the numbers in the question,
  earns full reward. The model learns that only the last line matters.
- **The format reward saturates, then conflicts.** Learned in a few steps, after which the group mean
  cancels it. Its regex requires the answer line **last**, punishing self-verification, and accepts only
  bare digits (no negatives, fractions, decimals).
- **Exact matching gives false negatives.** Gold `1,000` vs generated `1000` — correct answers scored 0,
  which both corrupts the advantage and depresses our Task 7 accuracy.
- **Truncation is invisible.** Hitting the 300-token cap is indistinguishable from being wrong, and since
  reasoning earns nothing, $\beta=0$ leaves a standing incentive to shorten it.

**A richer design**, cheapest first:

1. **Normalise before comparing.** Strip commas, currency symbols, spaces, trailing zeros; compare with `Decimal`/sympy
   equivalence. Removes the false negatives without creating any new incentive — do this first, since every
   other measurement depends on it.
2. **Raise $G$ and drop zero-advantage prompts.** More completions per prompt makes informative groups more
   likely; filtering constant-reward prompts (with a difficulty curriculum) concentrates compute where
   gradient exists. Highest leverage here, and it costs only generation time.
3. **Capped closeness reward.** Partial credit for near-misses, e.g.
   $r_{\text{num}}=r_{\max}\exp(-|\hat y-y|/(|y|+1))$ with $r_{\max}\le0.3$ while exact match keeps 2.0. The
   cap matters: uncapped, it is hackable by emitting a number of plausible magnitude.
4. **Dense step-level signal** — the real fix for credit assignment. Either parse each `a op b = c` and
   verify it with a calculator (still a program, so still unhackable); or score a prefix by the empirical
   solve rate of $k$ continuations (Math-Shepherd, no human step labels); or train a PRM
   (*Let's Verify Step by Step*) — strongest, but it reintroduces a learned reward model and so needs
   $\beta>0$.
5. **Program-of-thought execution.** Require a short program and run it: the interpreter gives a dense
   checkable signal and makes "right answer, invalid derivation" much harder.
6. **Self-consistency.** Reward agreement with the majority of $k$ samples — graded and label-free, but it
   reinforces confident mistakes.
7. **Small shaped terms.** A *graded* format reward (credit for the tags, then the exact final line) gives
   gradient from step one instead of saturating; add a truncation penalty and a repetition penalty. All must
   stay far below correctness, or the model optimises the shaping.
8. **Guards.** Smoothing a reward makes it easier to exploit: restore $\beta>0$, hold out a verifier never
   trained against, and watch training reward against held-out accuracy — divergence is the signature of
   over-optimisation and the point to stop.

<hr style="border:10px solid red"> </hr>
